# Score R3 candidates with bge-reranker-v2-m3 (Kaggle GPU)

Produces `scores_{split}.parquet` (qid, doc_id, score) and `meta_{split}.json` for R4
(protocol deviation 1: the reranker needs ~25 s/query on the benchmark CPU).

**Setup**
1. Add `artifacts/rerank/candidates_{split}.parquet` to the same private Kaggle Dataset that holds
   `corpus.parquet` (New Version → upload the file), and attach the dataset to this notebook.
2. Settings: Accelerator = **GPU T4 x2**, Internet = **on**.
3. Run all, then download the `rerank/` output folder into `artifacts/rerank/`.
4. Locally: `uv run python scripts/evaluate_rerank.py --tune` (dev).

`MODEL`, `REVISION` and `MAX_LENGTH` must match `configs/r4.yaml`.

In [ ]:
!pip install -q "sentence-transformers==6.2.0"

In [ ]:
import glob
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sentence_transformers
import torch
import transformers
from sentence_transformers import CrossEncoder

MODEL = "BAAI/bge-reranker-v2-m3"
REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
MAX_LENGTH = 512
BATCH_SIZE = 64
LATENCY_SAMPLE = 30  # queries timed one at a time with their top-20 (supplementary GPU latency)
OUT_DIR = Path("/kaggle/working/rerank")

assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings"
print(torch.cuda.get_device_name(0), "| sentence-transformers", sentence_transformers.__version__)

In [ ]:
(corpus_path,) = glob.glob("/kaggle/input/**/corpus.parquet", recursive=True)
contents = pd.read_parquet(corpus_path).set_index("doc_id")["contents"]
candidate_files = sorted(glob.glob("/kaggle/input/**/candidates_*.parquet", recursive=True))
assert candidate_files, "Upload candidates_{split}.parquet to the dataset"
print(f"{len(contents):,} articles; candidate files: {candidate_files}")

model = CrossEncoder(MODEL, revision=REVISION, max_length=MAX_LENGTH, device="cuda")
model.model.half()  # fp16 on GPU

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for path in candidate_files:
    split = re.search(r"candidates_(\w+)\.parquet", path).group(1)
    cands = pd.read_parquet(path)
    pairs = list(zip(cands["query"], contents.loc[cands["doc_id"]].tolist(), strict=True))

    start = time.perf_counter()
    scores = model.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=True)
    seconds = time.perf_counter() - start
    assert np.isfinite(scores).all()

    timings = []
    for qid in cands["qid"].drop_duplicates().head(LATENCY_SAMPLE):
        top = cands[(cands["qid"] == qid) & (cands["rank"] <= 20)]
        top_pairs = list(zip(top["query"], contents.loc[top["doc_id"]].tolist(), strict=True))
        t = time.perf_counter()
        model.predict(top_pairs, batch_size=20)
        timings.append(time.perf_counter() - t)

    out = cands[["qid", "doc_id", "rank"]].assign(score=scores.astype(float))
    out.to_parquet(OUT_DIR / f"scores_{split}.parquet", index=False)
    meta = {
        "model": MODEL,
        "revision": REVISION,
        "max_length": MAX_LENGTH,
        "split": split,
        "n_pairs": len(out),
        "seconds": seconds,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_latency_n20_p50_s": float(np.median(timings)),
        "gpu_latency_n20_p95_s": float(np.percentile(timings, 95)),
        "versions": {
            "sentence_transformers": sentence_transformers.__version__,
            "transformers": transformers.__version__,
            "torch": torch.__version__,
        },
    }
    (OUT_DIR / f"meta_{split}.json").write_text(json.dumps(meta, indent=2))
    print(split, f"{len(out):,} pairs in {seconds / 60:.1f} min", meta["gpu_latency_n20_p50_s"])